# LeetCode 1985 — Find the Kth Largest Integer in the Array bằng Python

Link đề: https://leetcode.com/problems/find-the-kth-largest-integer-in-the-array/ · Độ khó: **Medium** · Chủ đề: Array, String, Divide and Conquer, Sorting, Heap (Priority Queue), Quickselect

Skeleton LeetCode:

```python
class Solution:
    def kthLargestNumber(self, nums: List[str], k: int) -> str:
        ...
```

Các số được cho dưới dạng **xâu** dài tới 100 chữ số. Notebook đi từ **đếm số phần tử lớn hơn** (vét cạn `O(n²)`) → **sắp xếp với khóa `(độ dài, xâu)`** → **heap kích thước `k`** → **Quickselect** với phép so sánh tự viết.

## 1. Mục tiêu học

1. So sánh hai số nguyên dạng xâu **không có số 0 đứng đầu**: xâu dài hơn thì lớn hơn; cùng độ dài thì so theo thứ tự từ điển.
2. Dùng **khóa sắp xếp** `key=lambda s: (len(s), s)` thay vì đổi sang `int` (cách đúng cho mọi ngôn ngữ).
3. Cài đặt bài "phần tử lớn thứ `k`" bằng sắp xếp, **min-heap cỡ `k`** và **Quickselect** trung bình `O(n)`.
4. Hiểu "trùng lặp được đếm riêng" — không dùng `set`.
5. Kiểm thử đối chiếu với đổi sang `int` (Python có số lớn) trên dữ liệu ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Cho mảng xâu `nums` và số nguyên `k`. Mỗi xâu biểu diễn một số nguyên **không có số 0 đứng đầu**. Trả về xâu biểu diễn số **lớn thứ `k`** trong `nums`.

Lưu ý: các số trùng được **đếm riêng**. Ví dụ `["1","2","2"]`: lớn thứ nhất là `"2"`, lớn thứ hai cũng là `"2"`, lớn thứ ba là `"1"`.

**Ví dụ 1:** `nums = ["3","6","7","10"]`, `k = 4` → `"3"`.

**Ví dụ 2:** `nums = ["2","21","12","1"]`, `k = 3` → `"2"`.

**Ví dụ 3:** `nums = ["0","0"]`, `k = 2` → `"0"`.

**Ràng buộc:** `1 ≤ k ≤ nums.length ≤ 10⁴`, `1 ≤ nums[i].length ≤ 100`, chỉ gồm chữ số, không có số 0 đứng đầu.

In [1]:
import heapq
import random
import time
from functools import cmp_to_key
from typing import List

examples = [(["3", "6", "7", "10"], 4, "3"), (["2", "21", "12", "1"], 3, "2"), (["0", "0"], 2, "0")]

for nums, k, exp in examples:
    print(f"{nums}, k={k}: sắp theo giá trị {sorted(nums, key=int)} -> lớn thứ {k} = {sorted(nums, key=int)[-k]!r} (kỳ vọng {exp!r})")

['3', '6', '7', '10'], k=4: sắp theo giá trị ['3', '6', '7', '10'] -> lớn thứ 4 = '3' (kỳ vọng '3')
['2', '21', '12', '1'], k=3: sắp theo giá trị ['1', '2', '12', '21'] -> lớn thứ 3 = '2' (kỳ vọng '2')
['0', '0'], k=2: sắp theo giá trị ['0', '0'] -> lớn thứ 2 = '0' (kỳ vọng '0')


## 3. Kiến thức nền

### 3.1 So sánh xâu theo từ điển ≠ so sánh số

In [2]:
print('"10" < "3" theo từ điển?', "10" < "3", "| nhưng 10 < 3?", 10 < 3)
print(sorted(["3", "6", "7", "10"]), "<- sắp xâu thuần: sai thứ tự số")

"10" < "3" theo từ điển? True | nhưng 10 < 3? False
['10', '3', '6', '7'] <- sắp xâu thuần: sai thứ tự số


### 3.2 Khóa `(độ dài, xâu)`

Không có số 0 đứng đầu ⇒ số có nhiều chữ số hơn thì lớn hơn; cùng độ dài thì từ điển = số học. Tuple so sánh theo phần tử đầu trước.

In [3]:
key = lambda s: (len(s), s)
print(sorted(["3", "6", "7", "10", "21", "12"], key=key))

['3', '6', '7', '10', '12', '21']


### 3.3 Hàm so sánh và `cmp_to_key`

Muốn viết so sánh "kiểu C" trả về âm/0/dương: `functools.cmp_to_key`.

In [4]:
def cmp(a, b):
    if len(a) != len(b):
        return len(a) - len(b)
    return (a > b) - (a < b)

print(sorted(["21", "3", "100", "12"], key=cmp_to_key(cmp)))

['3', '12', '21', '100']


### 3.4 Heap với khóa: lưu tuple

`heapq` không nhận `key`; ta đẩy `(len(s), s)` để heap so sánh đúng. `heapq.nlargest(k, it, key=...)` làm sẵn việc này.

In [5]:
h = []
for s in ["2", "21", "12", "1"]:
    heapq.heappush(h, (len(s), s))
print("nhỏ nhất:", h[0], "| nlargest 2:", heapq.nlargest(2, ["2", "21", "12", "1"], key=lambda s: (len(s), s)))

nhỏ nhất: (1, '1') | nlargest 2: ['21', '12']


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `["2","21","12","1"]`. Theo khóa `(độ dài, xâu)`: `(1,"1") < (1,"2") < (2,"12") < (2,"21")`. Sắp giảm dần: `21, 12, 2, 1` ⇒ lớn thứ 3 là `"2"`.

Ví dụ 3: `["0","0"]`, `k = 2`: hai số `0` **đếm riêng** ⇒ lớn thứ nhất và thứ hai đều là `"0"`.

Câu hỏi "lớn thứ `k`" có cấu trúc chung (giống LeetCode 215): sắp xếp, heap cỡ `k`, hoặc Quickselect — chỉ khác **phép so sánh**.

In [6]:
nums = ["2", "21", "12", "1"]
desc = sorted(nums, key=lambda s: (len(s), s), reverse=True)
for r, s in enumerate(desc, 1):
    print(f"  lớn thứ {r}: {s}")

  lớn thứ 1: 21
  lớn thứ 2: 12
  lớn thứ 3: 2
  lớn thứ 4: 1


## 5. Cách 1 — Brute Force / baseline: đếm số phần tử lớn hơn

Với mỗi ứng viên `x`, đếm `greater = #{y > x}` và `ge = #{y ≥ x}`. `x` là lớn thứ `k` ⇔ `greater < k ≤ ge` (xử lý trùng đúng cách).

In [7]:
def greater(a, b):
    return (len(a), a) > (len(b), b)

def kth_brute(nums, k):
    for x in nums:
        g = sum(greater(y, x) for y in nums)
        e = sum(y == x for y in nums)
        if g < k <= g + e:
            return x

for nums_, k_, exp in examples:
    print(kth_brute(nums_, k_), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(n² · L)` với `L ≤ 100` là độ dài xâu — `10⁸·100` với `n = 10⁴`: quá chậm.

**Nút thắt:** so sánh lặp lại; sắp xếp một lần `O(n log n)` hoặc chọn thông minh hơn.

## 6. Ý tưởng thuật toán / bước đột phá

**So sánh đúng:** `a < b` (về số) ⇔ `(len(a), a) < (len(b), b)` — nhờ **không có số 0 đứng đầu** (với `"0"` duy nhất cũng đúng).

**Ba cách chọn:**
1. Sắp giảm dần theo khóa rồi lấy phần tử `k − 1`.
2. **Min-heap cỡ `k`**: duyệt, đẩy vào; nếu heap vượt `k` thì bỏ nhỏ nhất. Đỉnh heap cuối cùng là lớn thứ `k`.
3. **Quickselect**: chọn pivot, chia ba nhóm (lớn hơn / bằng / nhỏ hơn); **nếu** `k ≤ |lớn hơn|` **thì** tìm trong nhóm lớn hơn; **nếu** `k ≤ |lớn hơn| + |bằng|` **thì** đáp án là pivot; **ngược lại** tìm trong nhóm nhỏ hơn với `k` giảm.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp với khóa `(độ dài, xâu)`

In [8]:
def kth_sort(nums, k):
    return sorted(nums, key=lambda s: (len(s), s), reverse=True)[k - 1]

def kth_sort_int(nums, k):
    """Python có số nguyên lớn nên đổi int cũng chạy được (ngôn ngữ khác thì không)."""
    return sorted(nums, key=int, reverse=True)[k - 1]

for nums_, k_, exp in examples:
    print(kth_sort(nums_, k_), kth_sort_int(nums_, k_), "(kỳ vọng", exp, ")")

3 3 (kỳ vọng 3 )
2 2 (kỳ vọng 2 )
0 0 (kỳ vọng 0 )


**Độ phức tạp:** `O(n log n · L)`, bộ nhớ `O(n)`. Đổi sang `int` tốn `O(L)` mỗi số và chỉ dùng được ở ngôn ngữ có số lớn.

### 7.2 Cách 3 — Min-heap kích thước `k`

In [9]:
def kth_heap(nums, k):
    h = []
    for s in nums:
        item = (len(s), s)
        if len(h) < k:
            heapq.heappush(h, item)
        elif item > h[0]:
            heapq.heapreplace(h, item)          # thay phần tử nhỏ nhất trong top-k
    return h[0][1]

def kth_nlargest(nums, k):
    return heapq.nlargest(k, nums, key=lambda s: (len(s), s))[-1]

for nums_, k_, exp in examples:
    print(kth_heap(nums_, k_), kth_nlargest(nums_, k_), "(kỳ vọng", exp, ")")

3 3 (kỳ vọng 3 )
2 2 (kỳ vọng 2 )
0 0 (kỳ vọng 0 )


**Độ phức tạp:** `O(n log k · L)`, bộ nhớ `O(k)`. Tốt khi `k` nhỏ so với `n`.

### 7.3 Cách 4 — Quickselect (chia ba nhóm)

In [10]:
def kth_quickselect(nums, k, seed=0):
    rnd_local = random.Random(seed)
    items = [(len(s), s) for s in nums]
    while True:
        pivot = items[rnd_local.randrange(len(items))]
        big = [x for x in items if x > pivot]
        eq_count = sum(1 for x in items if x == pivot)
        if k <= len(big):
            items = big
        elif k <= len(big) + eq_count:
            return pivot[1]
        else:
            k -= len(big) + eq_count
            items = [x for x in items if x < pivot]

for nums_, k_, exp in examples:
    print(kth_quickselect(nums_, k_), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp:** trung bình `O(n · L)` (mỗi vòng giữ lại kỳ vọng một phần cố định), xấu nhất `O(n² · L)` (pivot tệ liên tục — hiếm với pivot ngẫu nhiên). Chia **ba** nhóm giúp xử lý nhiều phần tử trùng mà không lặp vô hạn.

### 7.4 Cách 5 — Gom theo độ dài (bucket theo số chữ số)

Vì độ dài ≤ 100, gom các xâu vào **101 xô theo độ dài**. Duyệt xô từ dài nhất xuống: nếu xô có `c` phần tử và `k > c` thì `k -= c` và bỏ qua cả xô (mọi số trong xô này đều lớn hơn các xô sau); ngược lại đáp án nằm **trong xô này** — chỉ cần sắp riêng xô đó theo từ điển.

Ưu điểm: chỉ sắp **một** xô, các xô khác chỉ đếm. Với dữ liệu nhiều độ dài khác nhau, thực tế rất nhanh.

In [11]:
def kth_bucket(nums, k):
    buckets = [[] for _ in range(101)]
    for s in nums:
        buckets[len(s)].append(s)
    for L in range(100, 0, -1):
        b = buckets[L]
        if k > len(b):
            k -= len(b)                       # cả xô lớn hơn đáp án
        else:
            return sorted(b, reverse=True)[k - 1]   # cùng độ dài: từ điển = số học

for nums_, k_, exp in examples:
    print(kth_bucket(nums_, k_), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [12]:
def trace_bucket(nums, k):
    buckets = {}
    for s in nums:
        buckets.setdefault(len(s), []).append(s)
    for L in sorted(buckets, reverse=True):
        b = buckets[L]
        if k > len(b):
            print(f"  độ dài {L}: {b} ({len(b)} số) -> k = {k} > {len(b)}, bỏ cả xô, k còn {k - len(b)}")
            k -= len(b)
        else:
            sb = sorted(b, reverse=True)
            print(f"  độ dài {L}: {b} -> sắp giảm {sb}, lấy vị trí {k} = {sb[k - 1]!r}")
            return sb[k - 1]

_ = trace_bucket(["7", "10", "3", "6", "100", "9", "42", "1000"], 4)

  độ dài 4: ['1000'] (1 số) -> k = 4 > 1, bỏ cả xô, k còn 3
  độ dài 3: ['100'] (1 số) -> k = 3 > 1, bỏ cả xô, k còn 2
  độ dài 2: ['10', '42'] -> sắp giảm ['42', '10'], lấy vị trí 2 = '10'


**Độ phức tạp:** `O(n · L)` để gom + sắp một xô `O(m log m · L)` với `m` là cỡ xô chứa đáp án. Xấu nhất (mọi số cùng độ dài) bằng cách 2.

### 7.5 Biến thể của cách 2 — sắp với hàm so sánh `cmp_to_key`

Cách "kiểu C++/Java" — viết comparator rồi sắp. Chậm hơn khóa tuple trong Python (mỗi lần so sánh gọi hàm Python) nhưng là cách tư duy chung khi ngôn ngữ không có khóa tuple.

In [13]:
def kth_cmp(nums, k):
    return sorted(nums, key=cmp_to_key(cmp), reverse=True)[k - 1]

for nums_, k_, exp in examples:
    print(kth_cmp(nums_, k_), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


### 7.6 Kiểm chứng khóa `(len, s)` là đơn điệu

Trước khi dựa vào khóa, kiểm bằng thực nghiệm: với mọi cặp số ngẫu nhiên (kể cả `0` và số rất dài), thứ tự theo khóa trùng thứ tự theo `int`.

In [14]:
rnd_chk = random.Random(7)
pool = ["0"] + [str(rnd_chk.randint(1, 10 ** rnd_chk.randint(1, 30))) for _ in range(400)]
bad = 0
for a in pool:
    for b in pool[:80]:
        if ((len(a), a) < (len(b), b)) != (int(a) < int(b)):
            bad += 1
print("số cặp sai thứ tự:", bad, "trên", len(pool) * 80, "cặp")
assert bad == 0

số cặp sai thứ tự: 0 trên 32080 cặp


## 8. TRACE từng bước

Trace cách 3 (heap) trên ví dụ 1 `["3","6","7","10"]`, `k = 4`, và trên `["7","10","3","6","100","9"]`, `k = 2` (ví dụ tự dựng):

In [15]:
def trace_heap(nums, k):
    h = []
    for s in nums:
        item = (len(s), s)
        if len(h) < k:
            heapq.heappush(h, item); act = "đẩy vào (heap chưa đủ k)"
        elif item > h[0]:
            old = heapq.heapreplace(h, item); act = f"thay {old[1]!r} (nhỏ nhất của top-k)"
        else:
            act = "bỏ qua (không vào top-k)"
        print(f"  {s!r:>6}: {act:<32} top-k = {sorted(x[1] for x in h)} | đỉnh = {h[0][1]!r}")
    return h[0][1]

print("=>", trace_heap(["3", "6", "7", "10"], 4))
print("=>", trace_heap(["7", "10", "3", "6", "100", "9"], 2))

     '3': đẩy vào (heap chưa đủ k)         top-k = ['3'] | đỉnh = '3'
     '6': đẩy vào (heap chưa đủ k)         top-k = ['3', '6'] | đỉnh = '3'
     '7': đẩy vào (heap chưa đủ k)         top-k = ['3', '6', '7'] | đỉnh = '3'
    '10': đẩy vào (heap chưa đủ k)         top-k = ['10', '3', '6', '7'] | đỉnh = '3'
=> 3
     '7': đẩy vào (heap chưa đủ k)         top-k = ['7'] | đỉnh = '7'
    '10': đẩy vào (heap chưa đủ k)         top-k = ['10', '7'] | đỉnh = '7'
     '3': bỏ qua (không vào top-k)         top-k = ['10', '7'] | đỉnh = '7'
     '6': bỏ qua (không vào top-k)         top-k = ['10', '7'] | đỉnh = '7'
   '100': thay '7' (nhỏ nhất của top-k)    top-k = ['10', '100'] | đỉnh = '10'
     '9': bỏ qua (không vào top-k)         top-k = ['10', '100'] | đỉnh = '10'
=> 10


In [16]:
def trace_quickselect(nums, k, seed=1):
    rnd_local = random.Random(seed)
    items = [(len(s), s) for s in nums]
    while True:
        pivot = items[rnd_local.randrange(len(items))]
        big = [x for x in items if x > pivot]
        eq = sum(1 for x in items if x == pivot)
        small = [x for x in items if x < pivot]
        print(f"  k={k}, pivot={pivot[1]!r}: lớn hơn {[x[1] for x in big]}, bằng {eq}, nhỏ hơn {[x[1] for x in small]}")
        if k <= len(big):
            items = big
        elif k <= len(big) + eq:
            print("  => đáp án", pivot[1]); return pivot[1]
        else:
            k -= len(big) + eq; items = small

_ = trace_quickselect(["2", "21", "12", "1", "21", "5"], 3)

  k=3, pivot='21': lớn hơn [], bằng 2, nhỏ hơn ['2', '12', '1', '5']
  k=1, pivot='2': lớn hơn ['12', '5'], bằng 1, nhỏ hơn ['1']
  k=1, pivot='5': lớn hơn ['12'], bằng 1, nhỏ hơn []
  k=1, pivot='12': lớn hơn [], bằng 1, nhỏ hơn []
  => đáp án 12


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def kthLargestNumber(self, nums: List[str], k: int) -> str:
        h = []                                   # min-heap chứa top-k theo (độ dài, xâu)
        for s in nums:
            item = (len(s), s)
            if len(h) < k:
                heapq.heappush(h, item)
            elif item > h[0]:
                heapq.heapreplace(h, item)
        return h[0][1]

for nums_, k_, exp in examples:
    assert Solution().kthLargestNumber(nums_, k_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Phép so sánh.** Hai xâu số không có số 0 đứng đầu: nếu `len(a) < len(b)` thì `a < 10^{len(a)} ≤ b`; nếu cùng độ dài, so sánh từ điển từng chữ số trùng với so sánh giá trị (chữ số đầu tiên khác nhau quyết định). ⇒ `(len, s)` là khóa **đơn điệu** theo giá trị số; các số bằng nhau có xâu giống hệt.

**Heap cỡ `k`.** Bất biến: sau khi duyệt `t` phần tử, heap chứa `min(t, k)` phần tử lớn nhất (đếm cả trùng) trong `t` phần tử đó. Phần tử mới chỉ vào heap khi lớn hơn phần tử nhỏ nhất của top-`k` (và thay nó). Cuối cùng đỉnh heap (nhỏ nhất trong top-`k`) là lớn thứ `k`.

**Quickselect.** Gọi `G`, `E`, `S` là các nhóm lớn hơn/bằng/nhỏ hơn pivot. Lớn thứ `k` nằm trong `G` nếu `k ≤ |G|`, bằng pivot nếu `|G| < k ≤ |G| + |E|`, ngược lại là lớn thứ `k − |G| − |E|` của `S`. Mỗi vòng tập giảm ít nhất 1 phần tử (pivot bị loại khỏi `G` và `S`) ⇒ dừng.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `L ≤ 100` độ dài xâu tối đa.

| Cách | Time | Space |
|---|---|---|
| 1. Đếm lớn hơn | `O(n² L)` | `O(1)` |
| 2. Sắp xếp | `O(n log n · L)` | `O(n)` |
| 3. Heap cỡ `k` | `O(n log k · L)` | `O(k)` |
| 4. Quickselect | trung bình `O(n L)`, xấu nhất `O(n² L)` | `O(n)` |
| 5. Xô theo độ dài | `O(n L + m log m · L)` | `O(n)` |
| 2b. `cmp_to_key` | `O(n log n · L)` (hằng số lớn) | `O(n)` |

In [18]:
rnd = random.Random(1985)
def rand_num(rnd):
    L = rnd.randint(1, 100)
    return "0" if L == 1 and rnd.random() < 0.1 else str(rnd.randint(1, 9)) + ''.join(rnd.choice("0123456789") for _ in range(L - 1))

big = [rand_num(rnd) for _ in range(10**4)]
for name, fn in [("sort", kth_sort), ("sort_int", kth_sort_int), ("heap", kth_heap), ("nlargest", kth_nlargest), ("quickselect", kth_quickselect), ("bucket", kth_bucket), ("cmp", kth_cmp), ("Solution", Solution().kthLargestNumber)]:
    t0 = time.perf_counter(); r = fn(big, 5000); dt = time.perf_counter() - t0
    print(f"{name:<12}: độ dài kết quả {len(r):>3}  {dt * 1000:7.1f} ms")

sort        : độ dài kết quả  50      3.5 ms
sort_int    : độ dài kết quả  50      3.0 ms
heap        : độ dài kết quả  50      2.4 ms
nlargest    : độ dài kết quả  50      5.1 ms
quickselect : độ dài kết quả  50      2.6 ms
bucket      : độ dài kết quả  50      0.4 ms
cmp         : độ dài kết quả  50      9.4 ms
Solution    : độ dài kết quả  50      2.1 ms


## 12. Bộ test

In [19]:
ALL = {
    "sort": kth_sort,
    "sort_int": kth_sort_int,
    "heap": kth_heap,
    "nlargest": kth_nlargest,
    "quickselect": kth_quickselect,
    "bucket": kth_bucket,
    "cmp": kth_cmp,
    "Solution": lambda a, k: Solution().kthLargestNumber(a, k),
}

def check(nums, k, exp=None, brute=True):
    ref = kth_brute(nums, k) if brute else kth_sort_int(nums, k)
    for name, fn in ALL.items():
        got = fn(list(nums), k)
        assert got == ref, (name, nums, k, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

for nums_, k_, exp in examples:
    check(nums_, k_, exp)
print("Ví dụ đề: OK")

check(["1", "2", "2"], 1, "2"); check(["1", "2", "2"], 2, "2"); check(["1", "2", "2"], 3, "1")   # đếm trùng
check(["9", "10", "99", "100"], 2, "99")
check(["5"], 1, "5")
check(["1" * 100, "9" * 99], 1, "1" * 100)                                                       # dài hơn thì lớn hơn
print("Edge case: OK")

rnd = random.Random(1999)
for _ in range(2000):
    n = rnd.randint(1, 12)
    nums_ = [str(rnd.randint(0, 120)) for _ in range(n)]
    check(nums_, rnd.randint(1, n))
print("2000 mảng nhỏ (nhiều trùng) so với đếm vét cạn: OK")

for _ in range(20):
    n = rnd.randint(1, 3000)
    nums_ = [rand_num(rnd) for _ in range(n)]
    check(nums_, rnd.randint(1, n), brute=False)
print("20 mảng cỡ lớn (số tới 100 chữ số) so với đổi int: OK")

Ví dụ đề: OK
Edge case: OK


2000 mảng nhỏ (nhiều trùng) so với đếm vét cạn: OK


20 mảng cỡ lớn (số tới 100 chữ số) so với đổi int: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp xếp xâu theo thứ tự từ điển

In [20]:
def kth_bug_lex(nums, k):
    return sorted(nums, reverse=True)[k - 1]                # SAI: "3" > "10" theo từ điển

nums_, k_ = ["3", "6", "7", "10"], 4
print("sai :", kth_bug_lex(nums_, k_), "| đúng:", Solution().kthLargestNumber(nums_, k_))

sai : 10 | đúng: 3


**Sửa:** khóa `(len(s), s)`.

### Lỗi 2 — Loại trùng bằng `set`

In [21]:
def kth_bug_set(nums, k):
    u = sorted(set(nums), key=lambda s: (len(s), s), reverse=True)
    return u[k - 1] if k <= len(u) else None                # SAI: trùng phải đếm riêng

for nums_, k_ in [(["0", "0"], 2), (["1", "2", "2"], 2)]:
    print(f"{nums_}, k={k_}: sai = {kth_bug_set(nums_, k_)}, đúng = {Solution().kthLargestNumber(nums_, k_)}")

['0', '0'], k=2: sai = None, đúng = 0
['1', '2', '2'], k=2: sai = 1, đúng = 2


**Sửa:** không loại trùng (đề nói rõ "đếm riêng").

### Lỗi 3 — Lấy chỉ số sai chiều (`[k]` hoặc `[-k]` trên mảng giảm dần)

In [22]:
def kth_bug_index(nums, k):
    return sorted(nums, key=lambda s: (len(s), s))[k - 1]   # SAI: mảng TĂNG dần -> phần tử nhỏ thứ k

nums_, k_ = ["2", "21", "12", "1"], 1
print("sai :", kth_bug_index(nums_, k_), "| đúng:", Solution().kthLargestNumber(nums_, k_))

sai : 1 | đúng: 21


**Sửa:** sắp **giảm** rồi `[k − 1]`, hoặc sắp tăng rồi `[-k]`.

### Lỗi 4 — Quickselect chia hai nhóm với pivot trùng ⇒ lặp vô hạn

In [23]:
def kth_bug_quickselect(nums, k, max_rounds=50):
    items = [(len(s), s) for s in nums]
    rounds = 0
    while rounds < max_rounds:                                # giới hạn vòng chỉ để demo không treo
        rounds += 1
        pivot = items[0]
        ge = [x for x in items if x >= pivot]                 # SAI: nhóm >= luôn chứa pivot và mọi phần tử bằng
        lt = [x for x in items if x < pivot]
        if k <= len(ge):
            if len(ge) == len(items):
                continue                                      # không thu nhỏ được -> lặp mãi
            items = ge
        else:
            k -= len(ge); items = lt
    return None

print("chia hai nhóm với ['7','7','7'], k=2:", kth_bug_quickselect(["7", "7", "7"], 2), "| đúng:", Solution().kthLargestNumber(["7", "7", "7"], 2))

chia hai nhóm với ['7','7','7'], k=2: None | đúng: 7


**Vì sao sai:** khi mọi phần tử bằng pivot, nhóm `>=` không nhỏ đi ⇒ không bao giờ dừng (ở đây bị giới hạn vòng nên trả `None`). **Sửa:** chia **ba** nhóm (lớn hơn / bằng / nhỏ hơn) như cách 4.

### Lỗi 5 — Đổi sang số nguyên 64-bit (thói quen từ C++/Java)

Python có số nguyên lớn nên `int(s)` luôn đúng; nhưng ở C++/Java, `stoll`/`Long.parseLong` tràn khi số > `2⁶³ − 1` (19 chữ số). Mô phỏng phép đổi 64-bit có dấu bằng cách lấy modulo `2⁶⁴`:

In [24]:
def to_int64(s):
    v = int(s) % (1 << 64)
    return v - (1 << 64) if v >= (1 << 63) else v      # giống tràn số có dấu 64-bit

def kth_bug_int64(nums, k):
    return sorted(nums, key=to_int64, reverse=True)[k - 1]

nums_ = ["9223372036854775808", "5", "123"]             # 2^63 -> tràn thành số âm
print("sai :", kth_bug_int64(nums_, 1), "| đúng:", Solution().kthLargestNumber(nums_, 1))
print("giá trị sau tràn:", to_int64("9223372036854775808"))

sai : 123 | đúng: 9223372036854775808
giá trị sau tràn: -9223372036854775808


**Sửa:** so sánh bằng `(len, s)` — không bao giờ tràn, chạy được ở mọi ngôn ngữ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Số lớn dạng xâu **không có số 0 đứng đầu** ⇒ so sánh bằng `(len, s)`.
- "Phần tử lớn thứ `k`" ⇒ sắp xếp / heap cỡ `k` / Quickselect (LeetCode 215).
- Python: `heapq.nlargest(k, nums, key=...)[-1]` là một dòng.
- Đếm trùng riêng ⇒ không dùng `set`.
- Quickselect: chia **ba** nhóm để an toàn với trùng lặp.

## 15. Biến thể của bài

1. **LeetCode 215 — Kth Largest Element in an Array:** số nguyên thường.
2. **Có số 0 đứng đầu** (ví dụ `"007"`): chuẩn hóa `s.lstrip('0') or '0'` trước khi so sánh.
3. **Số âm dạng xâu**: so sánh theo dấu trước, rồi đảo chiều với số âm.
4. **Lớn thứ `k` trong luồng dữ liệu** (LeetCode 703): duy trì heap cỡ `k`.
5. **LeetCode 179 — Largest Number:** sắp xâu với phép so sánh `a + b` vs `b + a`.

In [25]:
def kth_with_leading_zeros(nums, k):
    norm = lambda s: s.lstrip('0') or '0'
    return sorted(nums, key=lambda s: (len(norm(s)), norm(s)), reverse=True)[k - 1]

def kth_signed(nums, k):
    def key(s):
        if s.startswith('-'):
            d = s[1:]
            return (0, -len(d), tuple(-int(c) for c in d))      # số âm: dài hơn thì nhỏ hơn
        return (1, len(s), s)
    return sorted(nums, key=key, reverse=True)[k - 1]

print("có số 0 đầu:", kth_with_leading_zeros(["007", "10", "0009"], 1))
vals = ["-5", "3", "-12", "0", "-1"]
print("có số âm:", [kth_signed(vals, k) for k in range(1, 6)], "| kiểm:", sorted(vals, key=int, reverse=True))

có số 0 đầu: 10
có số âm: ['3', '0', '-1', '-5', '-12'] | kiểm: ['3', '0', '-1', '-5', '-12']


**Biến thể — top-`k` lớn nhất (trả cả danh sách, giảm dần):**

In [26]:
def top_k_numbers(nums, k):
    return heapq.nlargest(k, nums, key=lambda s: (len(s), s))

print(top_k_numbers(["3", "6", "7", "10", "100", "42"], 3))
rnd_v = random.Random(15)
for _ in range(300):
    a = [str(rnd_v.randint(0, 999)) for _ in range(rnd_v.randint(1, 15))]
    k = rnd_v.randint(1, len(a))
    assert top_k_numbers(a, k) == sorted(a, key=int, reverse=True)[:k]
print("top_k_numbers OK")

['100', '42', '10']
top_k_numbers OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đếm lớn hơn | `greater < k ≤ ge` | `O(n² L)` | `O(1)` | kiểm thử |
| 2. Sắp xếp | khóa `(len, s)` | `O(n log n L)` | `O(n)` | ngắn gọn |
| 3. Heap cỡ `k` | giữ top-k | `O(n log k L)` | `O(k)` | **nộp bài**, `k` nhỏ |
| 4. Quickselect | chia ba nhóm | TB `O(n L)` | `O(n)` | nhanh trung bình |
| 5. Xô theo độ dài | đếm xô, sắp một xô | `O(nL + m log m L)` | `O(n)` | nhiều độ dài khác nhau |
| 2b. `cmp_to_key` | comparator | `O(n log n L)` | `O(n)` | tư duy C++/Java |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().kthLargestNumber(["623986800","3","887298","695","794","6888794705","269409","59930972","723091307","726368","8028385786","378585"], 11)`?

**Bài 2 (tính tay):** Sắp giảm dần theo khóa `(độ dài, xâu)` cho `["10","9","100","99","1000"]`.

**Bài 3 (điền TODO):** Hoàn thiện `kth_smallest_number(nums, k)` — số **nhỏ thứ `k`**.

**Bài 4 (tự viết lại):** Viết lại Quickselect **tại chỗ** (phân hoạch 3 đường Dutch National Flag trên list) thay vì tạo list mới.

**Bài 5 (mini challenge):** Viết `KthLargestStream(k, nums)` với phương thức `add(s)` trả về số lớn thứ `k` hiện tại (các xâu số), mô phỏng LeetCode 703 và kiểm bằng sắp xếp lại sau mỗi lần thêm.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def kth_smallest_number(nums, k):
    # TODO: sắp tăng theo (len, s) rồi [k-1], hoặc max-heap cỡ k
    pass

# Bài 4 — TODO
def kth_quickselect_inplace(nums, k):
    # TODO: lo, hi; phân hoạch 3 đường quanh pivot theo thứ tự GIẢM dần
    pass

# Bài 5 — TODO
class KthLargestStream:
    def __init__(self, k, nums):
        # TODO: heap cỡ k
        pass
    def add(self, s):
        # TODO
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Có 12 số, `k = 11` ⇒ nhỏ thứ 2.
- Mức 2: Số nhỏ nhất là `"3"`.
- Mức 3: Nhỏ thứ hai là `"695"`.

**Bài 2**
- Mức 1: Độ dài: `1000` (4), `100` (3), `10`/`99` (2), `9` (1).
- Mức 2: Cùng độ dài 2: `"99" > "10"`.
- Mức 3: `1000, 100, 99, 10, 9`.

**Bài 3**
- Mức 1: Nhỏ thứ `k` = lớn thứ `n − k + 1`.
- Mức 2: Hoặc sắp tăng.
- Mức 3: Kiểm bằng `sorted(..., key=int)`.

**Bài 4**
- Mức 1: Ba vùng: `> pivot`, `== pivot`, `< pivot`.
- Mức 2: Con trỏ `lt`, `i`, `gt`.
- Mức 3: Thu hẹp `[lo, hi]` theo vùng chứa `k`.

**Bài 5**
- Mức 1: Heap tối thiểu cỡ `k` lưu `(len, s)`.
- Mức 2: `add`: đẩy vào, nếu vượt `k` thì pop.
- Mức 3: Đỉnh heap là đáp án.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
b1 = ["623986800", "3", "887298", "695", "794", "6888794705", "269409", "59930972", "723091307", "726368", "8028385786", "378585"]
r1 = Solution().kthLargestNumber(b1, 11)
print("Bài 1:", r1)
assert r1 == "695"

# Bài 2
r2 = sorted(["10", "9", "100", "99", "1000"], key=lambda s: (len(s), s), reverse=True)
print("Bài 2:", r2)
assert r2 == ["1000", "100", "99", "10", "9"]

# Bài 3
def kth_smallest_number_sol(nums, k):
    return sorted(nums, key=lambda s: (len(s), s))[k - 1]

rnd = random.Random(3)
for _ in range(500):
    a = [str(rnd.randint(0, 500)) for _ in range(rnd.randint(1, 20))]
    k = rnd.randint(1, len(a))
    assert kth_smallest_number_sol(a, k) == sorted(a, key=int)[k - 1]
print("Bài 3 OK")

Bài 1: 695
Bài 2: ['1000', '100', '99', '10', '9']
Bài 3 OK


In [29]:
# Bài 4
def kth_quickselect_inplace_sol(nums, k):
    a = [(len(s), s) for s in nums]
    lo, hi = 0, len(a) - 1
    rnd_local = random.Random(0)
    while True:
        p = a[rnd_local.randint(lo, hi)]
        lt, i, gt = lo, lo, hi                 # [lo, lt) > p, [lt, i) == p, (gt, hi] < p
        while i <= gt:
            if a[i] > p:
                a[lt], a[i] = a[i], a[lt]; lt += 1; i += 1
            elif a[i] < p:
                a[i], a[gt] = a[gt], a[i]; gt -= 1
            else:
                i += 1
        idx = lo + k - 1                       # vị trí cần tìm (theo thứ tự giảm) trong đoạn
        if idx < lt:
            hi = lt - 1
        elif idx <= gt:
            return p[1]
        else:
            k -= (gt + 1 - lo); lo = gt + 1

# Bài 5
class KthLargestStreamSol:
    def __init__(self, k, nums):
        self.k = k; self.h = []
        for s in nums:
            self.add(s)
    def add(self, s):
        heapq.heappush(self.h, (len(s), s))
        if len(self.h) > self.k:
            heapq.heappop(self.h)
        return self.h[0][1] if len(self.h) == self.k else None

rnd = random.Random(5)
for _ in range(500):
    a = [str(rnd.randint(0, 300)) for _ in range(rnd.randint(1, 30))]
    k = rnd.randint(1, len(a))
    assert kth_quickselect_inplace_sol(a, k) == kth_sort_int(a, k)
for _ in range(100):
    k = rnd.randint(1, 5); init = [str(rnd.randint(0, 50)) for _ in range(k)]
    st = KthLargestStreamSol(k, init); cur = list(init)
    for _ in range(20):
        s = str(rnd.randint(0, 50)); cur.append(s)
        assert st.add(s) == sorted(cur, key=int, reverse=True)[k - 1]
print("Bài 4, Bài 5 OK")

Bài 4, Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template so sánh số dạng xâu + heap cỡ k:**

```python
h = []
for s in nums:
    item = (len(s), s)            # khóa đúng thứ tự số (không có 0 đứng đầu)
    if len(h) < k: heapq.heappush(h, item)
    elif item > h[0]: heapq.heapreplace(h, item)
return h[0][1]
```

**Một dòng:** `heapq.nlargest(k, nums, key=lambda s: (len(s), s))[-1]`.

**Checklist:**
- [ ] So sánh theo `(len, s)`, không theo từ điển thuần?
- [ ] Không loại trùng?
- [ ] Đúng chiều chỉ số (lớn thứ `k`)?
- [ ] Quickselect chia ba nhóm?
- [ ] Đã so với đổi `int` trên dữ liệu ngẫu nhiên?
- [ ] Chữ ký đúng: `def kthLargestNumber(self, nums: List[str], k: int) -> str`?